# COVID-19 Positive Case Forecasting: XGBoost vs Chronos-2 — Google Colab version

**How to run:** `Runtime → Run all`. When the upload button appears in step 0.2, choose **one or both** of:

| Upload | What happens |
|---|---|
| `covid_forecast.zip` (the project zip) | Unzipped; its code, data and templates are used |
| `covid_data.xlsx` (or any Excel file with the same columns) | Used as the data; **overrides** the Excel inside the zip |
| A zip that only contains the Excel file | The Excel is extracted and used |

Only the Excel file is needed: this notebook includes a copy of `pipeline.py` and the Streamlit `app.py`, and writes them if they aren't in an uploaded zip (or are older versions).

**Google Drive instead of uploading:** set `SOURCE = "drive"` in step 0.2 and put the files in `DRIVE_FOLDER`.

The notebook compares XGBoost with **Chronos-2** (a pretrained forecasting model) in a backtest and picks the best one automatically. A GPU runtime (`Runtime → Change runtime type → T4 GPU`) makes Chronos-2 faster but isn't required.

At the end, step 7 downloads the trained model and outputs as a zip, and step 8 starts the Streamlit web page with a public localtunnel link.

## 0. Colab setup

In [ ]:
#@title 0.1 Install / check packages
import sys, subprocess, importlib, importlib.util, os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

need = {"xgboost": "xgboost>=2.0", "openpyxl": "openpyxl", "sklearn": "scikit-learn",
        "streamlit": "streamlit", "plotly": "plotly", "chronos": "chronos-forecasting>=2.0"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import xgboost, pandas
print(f"Running in Colab: {IN_COLAB} | xgboost {xgboost.__version__} | pandas {pandas.__version__}")

In [ ]:
#@title 0.2 Upload the project zip and/or the Excel data
SOURCE = "upload"   #@param ["upload", "drive"]
DRIVE_FOLDER = "/content/drive/MyDrive/covid_forecast"  #@param {type:"string"}

import shutil, zipfile, glob, io

WORK = "/content" if IN_COLAB else os.path.abspath(os.environ.get("COVID_WORKDIR", "."))
PROJECT = os.path.join(WORK, "covid_forecast")
INBOX = os.path.join(WORK, "_uploads")
os.makedirs(INBOX, exist_ok=True)
EXCEL_EXT = (".xlsx", ".xlsm", ".xls")

# ---- 1. collect files ---------------------------------------------------
received = []
if SOURCE == "drive":
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    for p in sorted(glob.glob(os.path.join(DRIVE_FOLDER, "*"))):
        if p.lower().endswith((".zip",) + EXCEL_EXT):
            dst = os.path.join(INBOX, os.path.basename(p)); shutil.copy(p, dst); received.append(dst)
elif IN_COLAB:
    from google.colab import files
    print("Select covid_forecast.zip and/or covid_data.xlsx (you can pick both at once).")
    for name, data in files.upload().items():
        dst = os.path.join(INBOX, os.path.basename(name))
        with open(dst, "wb") as f:
            f.write(data)
        received.append(dst)
else:  # running locally: use files already in the inbox / working dir
    received = [p for p in glob.glob(os.path.join(INBOX, "*")) + glob.glob(os.path.join(WORK, "*"))
                if p.lower().endswith((".zip",) + EXCEL_EXT)]
print("Received:", [os.path.basename(p) for p in received] or "nothing")

# ---- 2. unpack zips, find the project and any Excel files ----------------
direct_excels, zipped_excels = [], []
for path in received:
    low = path.lower()
    if low.endswith(EXCEL_EXT):
        direct_excels.append(path)
    elif low.endswith(".zip"):
        if not zipfile.is_zipfile(path):
            print(f"  ! {os.path.basename(path)} is not a valid zip, skipped"); continue
        out = os.path.join(INBOX, os.path.splitext(os.path.basename(path))[0] + "_unzipped")
        shutil.rmtree(out, ignore_errors=True)
        with zipfile.ZipFile(path) as z:
            z.extractall(out)
        # the project root is the folder that holds pipeline.py
        roots = [os.path.dirname(p) for p in glob.glob(os.path.join(out, "**", "pipeline.py"), recursive=True)
                 if "__MACOSX" not in p]
        if roots:
            shutil.copytree(roots[0], PROJECT, dirs_exist_ok=True)
            print(f"  project code from {os.path.basename(path)}")
        zipped_excels += [p for p in glob.glob(os.path.join(out, "**", "*"), recursive=True)
                          if p.lower().endswith(EXCEL_EXT) and "__MACOSX" not in p
                          and not (roots and p.startswith(roots[0] + os.sep))]

In [ ]:
#@title 0.3 Write project files that weren't in an upload (embedded copies)
SOURCES = {
 "pipeline.py": "\"\"\"\nShared data-cleaning, feature-engineering and forecasting logic.\n\nUsed by both the Jupyter notebook (training) and the Streamlit app (serving),\nso the features the model sees at prediction time are built exactly the\nsame way as during training.\n\"\"\"\nfrom __future__ import annotations\n\nimport re\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------\n# 1. Cleaning\n# --------------------------------------------------------------------------\nBINARY_COLS = [\n    \"intubed\", \"pneumonia\", \"pregnancy\", \"diabetes\", \"copd\", \"asthma\",\n    \"inmsupr\", \"hypertension\", \"other_disease\", \"cardiovascular\", \"obesity\",\n    \"renal_chronic\", \"tobacco\", \"contact_other_covid\", \"icu\",\n]\nCOMORBIDITIES = [\n    \"diabetes\", \"copd\", \"asthma\", \"inmsupr\", \"hypertension\", \"other_disease\",\n    \"cardiovascular\", \"obesity\", \"renal_chronic\", \"tobacco\",\n]\nDATE_COLS = [\"entry_date\", \"date_symptoms\", \"date_died\"]\nSEX_MAP = {\"male\": \"Male\", \"males\": \"Male\", \"m\": \"Male\",\n           \"female\": \"Female\", \"femalee\": \"Female\", \"f\": \"Female\"}\nAGE_MIN, AGE_MAX = 0, 110\n\n\nREQUIRED_COLS = [\"id\", \"sex\", \"patient_type\", \"entry_date\", \"date_symptoms\", \"date_died\",\n                 \"age\", \"covid_res\"] + BINARY_COLS\n\n\ndef read_patient_excel(path: str) -> tuple[pd.DataFrame, pd.DataFrame | None]:\n    \"\"\"Read the patient sheet (and data dictionary if present) from an Excel file.\n\n    Uses the 'covid19_patient_data' sheet if it exists, otherwise the first sheet,\n    and checks that all columns the pipeline needs are present.\n    \"\"\"\n    xl = pd.ExcelFile(path)\n    sheet = \"covid19_patient_data\" if \"covid19_patient_data\" in xl.sheet_names else xl.sheet_names[0]\n    raw = pd.read_excel(xl, sheet_name=sheet, dtype={\"id\": str})\n    raw.columns = [str(c).strip().lower() for c in raw.columns]\n    missing = [c for c in REQUIRED_COLS if c not in raw.columns]\n    if missing:\n        raise ValueError(f\"Sheet '{sheet}' in {path} is missing columns: {missing}\")\n    dictionary = (pd.read_excel(xl, sheet_name=\"data_dictionary\")\n                  if \"data_dictionary\" in xl.sheet_names else None)\n    return raw, dictionary\n\n\ndef _fix_id(raw) -> str | None:\n    \"\"\"Repair IDs that Excel corrupted. Valid IDs are 6-char lowercase hex.\n\n    * '50969'    -> '050969'  (numeric-looking hex lost its leading zeros)\n    * '1.34E+08' -> None      (scientific notation: original value is lost)\n    \"\"\"\n    s = str(raw).strip().lower()\n    if re.fullmatch(r\"[0-9a-f]{6}\", s):\n        return s\n    if re.fullmatch(r\"\\d{1,5}\", s):\n        return s.zfill(6)\n    return None  # unrecoverable\n\n\ndef clean_data(raw: pd.DataFrame, verbose: bool = True) -> tuple[pd.DataFrame, dict]:\n    \"\"\"Return a cleaned copy of the patient-level data plus a log of fixes.\"\"\"\n    df = raw.copy()\n    log: dict[str, int] = {}\n\n    # Strip whitespace on all text columns\n    for c in df.select_dtypes(include=[\"object\", \"string\"]).columns:\n        df[c] = df[c].astype(\"string\").str.strip()\n\n    # --- IDs -------------------------------------------------------------\n    fixed = df[\"id\"].map(_fix_id)\n    log[\"id_zero_padded\"] = int(((fixed.notna()) & (fixed != df[\"id\"].str.lower())).sum())\n    bad = fixed.isna()\n    log[\"id_unrecoverable_regenerated\"] = int(bad.sum())\n    df[\"id_was_corrupt\"] = bad | (fixed != df[\"id\"].str.lower())\n    # give unrecoverable IDs a new, clearly-synthetic unique id\n    df.loc[bad, \"id\"] = [f\"gen{i:05d}\" for i in range(int(bad.sum()))]\n    df.loc[~bad, \"id\"] = fixed[~bad]\n    dup = df.duplicated(keep=\"first\")\n    log[\"exact_duplicate_rows_dropped\"] = int(dup.sum())\n    df = df[~dup]\n    log[\"duplicate_ids_after_fix\"] = int(df[\"id\"].duplicated().sum())\n\n    # --- Sex typos -------------------------------------------------------\n    before = df[\"sex\"].copy()\n    df[\"sex\"] = df[\"sex\"].str.lower().map(SEX_MAP)\n    log[\"sex_typos_fixed\"] = int((before != df[\"sex\"]).sum())\n\n    # --- Dates (dd-mm-yyyy) ----------------------------------------------\n    for c in DATE_COLS:\n        df[c] = pd.to_datetime(df[c], format=\"%d-%m-%Y\", errors=\"coerce\")\n    # symptoms can't start after admission -> swap would be a guess, so null it\n    bad_sym = df[\"date_symptoms\"] > df[\"entry_date\"]\n    log[\"symptoms_after_entry_nulled\"] = int(bad_sym.sum())\n    df.loc[bad_sym, \"date_symptoms\"] = pd.NaT\n    # death recorded before admission -> inconsistent, flag + null\n    bad_died = df[\"date_died\"] < df[\"entry_date\"]\n    log[\"died_before_entry_nulled\"] = int(bad_died.sum())\n    df[\"date_inconsistent\"] = bad_died | bad_sym\n    df.loc[bad_died, \"date_died\"] = pd.NaT\n\n    # --- Age outliers ----------------------------------------------------\n    bad_age = (df[\"age\"] < AGE_MIN) | (df[\"age\"] > AGE_MAX)\n    log[\"age_out_of_range_imputed\"] = int(bad_age.sum())\n    df[\"age_was_invalid\"] = bad_age\n    df.loc[bad_age, \"age\"] = np.nan\n    df[\"age\"] = df[\"age\"].fillna(df[\"age\"].median()).astype(int)\n\n    # --- Yes/No -> 1/0 with structural missingness handled ---------------\n    for c in BINARY_COLS:\n        df[c] = df[c].map({\"Yes\": 1, \"No\": 0}).astype(\"Float64\")\n    # pregnancy is not applicable for males\n    male = df[\"sex\"] == \"Male\"\n    df.loc[male & df[\"pregnancy\"].isna(), \"pregnancy\"] = 0\n    # intubation / ICU only recorded for inpatients -> outpatients = 0\n    outp = df[\"patient_type\"] == \"Outpatient\"\n    for c in [\"intubed\", \"icu\"]:\n        df.loc[outp & df[c].isna(), c] = 0\n    # remaining small gaps in comorbidities (~0.3%) -> assume absent (mode)\n    n_gap = int(df[COMORBIDITIES + [\"pneumonia\", \"pregnancy\"]].isna().sum().sum())\n    log[\"comorbidity_gaps_filled_with_no\"] = n_gap\n    for c in COMORBIDITIES + [\"pneumonia\", \"pregnancy\", \"intubed\", \"icu\"]:\n        df[c] = df[c].fillna(0).astype(int)\n    # contact_other_covid is 31% missing -> keep as explicit \"unknown\" (-1)\n    df[\"contact_other_covid\"] = df[\"contact_other_covid\"].fillna(-1).astype(int)\n\n    # --- Derived fields --------------------------------------------------\n    df[\"died\"] = df[\"date_died\"].notna().astype(int)\n    df[\"covid_res\"] = df[\"covid_res\"].astype(\"category\")\n    df[\"is_positive\"] = (df[\"covid_res\"] == \"Positive\").astype(int)\n\n    df = df.reset_index(drop=True)\n    if verbose:\n        for k, v in log.items():\n            print(f\"  {k:<36} {v:>6}\")\n    return df, log\n\n\n# --------------------------------------------------------------------------\n# 2. Daily time series\n# --------------------------------------------------------------------------\ndef build_daily(df: pd.DataFrame, start: str = \"2020-03-01\",\n                max_awaiting_share: float = 0.25) -> tuple[pd.DataFrame, pd.Timestamp]:\n    \"\"\"Aggregate to one row per entry date.\n\n    Columns: positive, negative, awaiting (results pending), tests,\n    awaiting_share, admissions (non-outpatients) and contacts (patients\n    reporting contact with another COVID case).\n\n    Returns the daily frame and the last *complete* date: trailing days where\n    more than `max_awaiting_share` of tests are still 'Awaiting Results' are\n    reporting-lag artefacts, not a real fall in cases, so the model is not\n    trained on them.\n    \"\"\"\n    t = pd.crosstab(df[\"entry_date\"], df[\"covid_res\"])\n    for c in [\"Positive\", \"Negative\", \"Awaiting Results\"]:\n        if c not in t:\n            t[c] = 0\n    t = t.asfreq(\"D\", fill_value=0)\n    daily = pd.DataFrame({\n        \"positive\": t[\"Positive\"],\n        \"negative\": t[\"Negative\"],\n        \"awaiting\": t[\"Awaiting Results\"],\n    })\n    daily[\"tests\"] = daily[\"positive\"] + daily[\"negative\"] + daily[\"awaiting\"]\n    daily[\"awaiting_share\"] = (daily[\"awaiting\"] / daily[\"tests\"].replace(0, np.nan)).fillna(0)\n\n    # extra daily counts used as model inputs (see EXOG_COLS)\n    is_admitted = df[\"patient_type\"].notna() & (df[\"patient_type\"] != \"Outpatient\")\n    extra = pd.DataFrame({\n        \"admissions\": is_admitted.astype(int).groupby(df[\"entry_date\"]).sum(),\n        \"contacts\": (df[\"contact_other_covid\"] == 1).astype(int).groupby(df[\"entry_date\"]).sum(),\n    })\n    daily = daily.join(extra.reindex(daily.index).fillna(0).astype(int))\n    daily = daily.loc[start:]\n\n    # walk back from the end while the tail is incomplete\n    complete = daily.index[-1]\n    for d in daily.index[::-1]:\n        if daily.at[d, \"awaiting_share\"] <= max_awaiting_share:\n            complete = d\n            break\n    daily.index.name = \"date\"\n    return daily, complete\n\n\n# --------------------------------------------------------------------------\n# 3. Features\n# --------------------------------------------------------------------------\nLAGS = [1, 2, 3, 7, 14]\nWINDOWS = [7, 14]\nBASE_FEATURES = ([f\"lag_{l}\" for l in LAGS]\n                 + [f\"roll_mean_{w}\" for w in WINDOWS]\n                 + [\"roll_std_7\", \"dow\", \"is_weekend\", \"trend_7\"])\n\n# Other daily counts used as inputs, each as yesterday's value and a 7-day mean.\n#   awaiting   - tests still pending on that entry date\n#   contacts   - patients reporting contact with another COVID case\n#   admissions - patients admitted (not outpatients)\nEXOG_COLS = [\"awaiting\", \"contacts\", \"admissions\"]\nEXOG_FEATURES = [f\"{c}_{k}\" for c in EXOG_COLS for k in (\"lag_1\", \"roll_7\")]\nFEATURES = BASE_FEATURES + EXOG_FEATURES\n\n\ndef make_features(y: pd.Series, exog: pd.DataFrame | None = None) -> pd.DataFrame:\n    \"\"\"Features for predicting y[t] using only information up to t-1.\n\n    `exog` is a daily frame holding EXOG_COLS (e.g. the output of build_daily).\n    Without it only BASE_FEATURES are returned.\n    \"\"\"\n    X = pd.DataFrame(index=y.index)\n    for l in LAGS:\n        X[f\"lag_{l}\"] = y.shift(l)\n    past = y.shift(1)\n    for w in WINDOWS:\n        X[f\"roll_mean_{w}\"] = past.rolling(w).mean()\n    X[\"roll_std_7\"] = past.rolling(7).std()\n    X[\"dow\"] = y.index.dayofweek\n    X[\"is_weekend\"] = (X[\"dow\"] >= 5).astype(int)\n    # week-over-week momentum of the 7-day mean\n    X[\"trend_7\"] = X[\"roll_mean_7\"] - past.shift(7).rolling(7).mean()\n    if exog is None:\n        return X[BASE_FEATURES]\n    ex_past = exog.reindex(y.index)[EXOG_COLS].astype(float).shift(1)\n    for c in EXOG_COLS:\n        X[f\"{c}_lag_1\"] = ex_past[c]\n        X[f\"{c}_roll_7\"] = ex_past[c].rolling(7).mean()\n    return X[FEATURES]\n\n\ndef recursive_forecast(model, history: pd.Series, horizon: int,\n                       exog: pd.DataFrame | None = None) -> pd.Series:\n    \"\"\"Multi-step forecast: each prediction is fed back as a lag for the next.\n\n    Only exog values up to the last day of `history` are used (no peeking).\n    Future exog values are unknown, so each is filled with the value from the\n    same weekday one week earlier (seasonal naive).\n    \"\"\"\n    y = history.astype(float).copy()\n    ex = None\n    if exog is not None:\n        ex = exog.loc[:y.index[-1], EXOG_COLS].astype(float).copy()\n    preds = {}\n    for _ in range(horizon):\n        nxt = y.index[-1] + pd.Timedelta(days=1)\n        y.loc[nxt] = np.nan\n        if ex is not None:\n            ex.loc[nxt] = ex.iloc[-7].values\n        x = make_features(y, ex).iloc[[-1]]\n        p = float(max(model.predict(x)[0], 0.0))\n        y.loc[nxt] = p\n        preds[nxt] = p\n    return pd.Series(preds, name=\"forecast\")\n\n\ndef one_step_predictions(model, y: pd.Series, exog: pd.DataFrame | None = None) -> pd.Series:\n    \"\"\"In-sample/backtest one-day-ahead predictions using actual lags.\"\"\"\n    X = make_features(y, exog)[model.features].dropna()\n    return pd.Series(np.clip(model.predict(X), 0, None), index=X.index, name=\"predicted\")\n\n\n# --------------------------------------------------------------------------\n# 4. Model\n# --------------------------------------------------------------------------\nimport json\nimport xgboost as xgb\n\nDEFAULT_PARAMS = dict(n_estimators=300, max_depth=2, learning_rate=0.05,\n                      subsample=0.9, colsample_bytree=0.9, min_child_weight=2,\n                      random_state=42)\nTRAIN_START = \"2020-03-20\"  # before this, daily positives are ~0 and uninformative\n\n\nclass LevelRelativeXGB:\n    \"\"\"XGBoost that predicts log(1+y) relative to the trailing 7-day mean.\n\n    Tree models can't extrapolate beyond the target range seen in training,\n    which is a problem for a rising epidemic curve. Modelling the log-ratio\n    to the recent level makes the target scale-free; the level is then\n    multiplied back in:  y_hat = expm1( f(X) + log1p(roll_mean_7) ).\n    \"\"\"\n\n    def __init__(self, features: list[str] | None = None, **params):\n        self.features = list(features or FEATURES)\n        self.params = {**DEFAULT_PARAMS, **params}\n        self.model = xgb.XGBRegressor(**self.params)\n        self._booster = None  # set when fitted or loaded\n\n    def fit(self, X: pd.DataFrame, y: pd.Series):\n        target = np.log1p(y) - np.log1p(X[\"roll_mean_7\"])\n        self._check(X)\n        self.model.fit(X[self.features], target)\n        self._booster = self.model.get_booster()\n        return self\n\n    def predict(self, X: pd.DataFrame) -> np.ndarray:\n        # predict through the raw Booster: works the same for fitted and loaded\n        # models, on every xgboost/scikit-learn version combination (e.g. Colab)\n        self._check(X)\n        r = self._booster.inplace_predict(X[self.features].astype(float))\n        return np.clip(np.expm1(r + np.log1p(X[\"roll_mean_7\"].values)), 0, None)\n\n    def forecast(self, history: pd.Series, horizon: int,\n                 exog: pd.DataFrame | None = None) -> pd.DataFrame:\n        \"\"\"Same interface as ChronosForecaster.forecast (no interval: lo/hi are NaN).\"\"\"\n        fc = recursive_forecast(self, history, horizon, exog)\n        return pd.DataFrame({\"forecast\": fc, \"lo\": np.nan, \"hi\": np.nan})\n\n    def _check(self, X: pd.DataFrame):\n        missing = [f for f in self.features if f not in X.columns]\n        if missing:\n            raise ValueError(f\"Missing features {missing}: pass `exog` (the daily frame) \"\n                             \"to make_features / recursive_forecast\")\n\n    @property\n    def feature_importances_(self):\n        gain = self._booster.get_score(importance_type=\"gain\")\n        s = pd.Series({f: gain.get(f, 0.0) for f in self.features})\n        return s / s.sum()\n\n    def save(self, path_prefix: str):\n        self._booster.save_model(f\"{path_prefix}.json\")\n        with open(f\"{path_prefix}_meta.json\", \"w\") as f:\n            json.dump({\"params\": self.params, \"features\": self.features,\n                       \"xgboost_version\": xgb.__version__}, f, indent=2)\n\n    @classmethod\n    def load(cls, path_prefix: str) -> \"LevelRelativeXGB\":\n        with open(f\"{path_prefix}_meta.json\") as f:\n            meta = json.load(f)\n        obj = cls(features=meta[\"features\"], **meta[\"params\"])\n        obj._booster = xgb.Booster()\n        obj._booster.load_model(f\"{path_prefix}.json\")\n        return obj\n\n\ndef train_model(y: pd.Series, exog: pd.DataFrame | None = None,\n                features: list[str] | None = None, **params) -> LevelRelativeXGB:\n    features = list(features or (FEATURES if exog is not None else BASE_FEATURES))\n    X = make_features(y, exog)[features].dropna().loc[TRAIN_START:]\n    return LevelRelativeXGB(features=features, **params).fit(X, y.loc[X.index])\n\n\n# --------------------------------------------------------------------------\n# 5. Chronos-2 (pretrained time-series foundation model, zero-shot)\n# --------------------------------------------------------------------------\nCHRONOS_MODEL_ID = \"amazon/chronos-2\"\n_CHRONOS_CACHE: dict = {}\n\n\ndef load_chronos(model_id: str = CHRONOS_MODEL_ID, device: str | None = None):\n    \"\"\"Load (once) and return a Chronos-2 pipeline. Needs `chronos-forecasting>=2.0`.\"\"\"\n    key = (model_id, device)\n    if key not in _CHRONOS_CACHE:\n        import torch\n        from chronos import Chronos2Pipeline\n        device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        _CHRONOS_CACHE[key] = Chronos2Pipeline.from_pretrained(model_id, device_map=device)\n    return _CHRONOS_CACHE[key]\n\n\nclass ChronosForecaster:\n    \"\"\"Zero-shot Chronos-2 forecast of daily positives.\n\n    Nothing is trained: the model reads the history up to the cut-off as\n    context, so every forecast is automatically an honest backtest.\n    With `use_covariates=True` the pending / contacts / admissions series are\n    passed as *past-only* covariates (known up to the cut-off, not after), so\n    no guessing of their future values is needed.\n    \"\"\"\n\n    QUANTILES = [0.1, 0.5, 0.9]\n\n    def __init__(self, use_covariates: bool = False, model_id: str = CHRONOS_MODEL_ID,\n                 pipeline=None):\n        self.use_covariates = use_covariates\n        self.model_id = model_id\n        self._pipeline = pipeline  # injected for tests; otherwise loaded lazily\n\n    @property\n    def pipeline(self):\n        if self._pipeline is None:\n            self._pipeline = load_chronos(self.model_id)\n        return self._pipeline\n\n    def forecast(self, history: pd.Series, horizon: int,\n                 exog: pd.DataFrame | None = None) -> pd.DataFrame:\n        \"\"\"Median forecast plus a 10-90% interval, indexed by future date.\"\"\"\n        y = history.astype(float)\n        ctx = pd.DataFrame({\"id\": \"positive\", \"timestamp\": y.index, \"target\": y.values})\n        if self.use_covariates:\n            if exog is None:\n                raise ValueError(\"use_covariates=True needs `exog` (the daily frame)\")\n            ex = exog.reindex(y.index)[EXOG_COLS].astype(float).ffill().fillna(0.0)\n            for c in EXOG_COLS:\n                ctx[c] = ex[c].values\n        pred = self.pipeline.predict_df(\n            ctx, prediction_length=horizon, quantile_levels=self.QUANTILES,\n            id_column=\"id\", timestamp_column=\"timestamp\", target=\"target\")\n        idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq=\"D\")\n        lo, med, hi = (pred[str(q)].to_numpy(float)[:horizon] for q in self.QUANTILES)\n        out = pd.DataFrame({\"forecast\": med, \"lo\": lo, \"hi\": hi}, index=idx).clip(lower=0)\n        out.index.name = \"date\"\n        return out\n\n    def config(self) -> dict:\n        return {\"kind\": \"chronos\", \"model_id\": self.model_id,\n                \"use_covariates\": self.use_covariates}\n\n\n# --------------------------------------------------------------------------\n# 6. Model registry: which model the app should serve\n# --------------------------------------------------------------------------\nMODEL_NAMES = {\n    \"xgb\": \"XGBoost (level-relative)\",\n    \"chronos\": \"Chronos-2\",\n    \"chronos_cov\": \"Chronos-2 + covariates\",\n}\n\n\ndef save_best(models_dir: str, best_name: str, comparison: list[dict]):\n    \"\"\"Record which model won the backtest (the XGBoost model is saved separately).\"\"\"\n    with open(f\"{models_dir}/best_model.json\", \"w\") as f:\n        json.dump({\"best\": best_name, \"comparison\": comparison}, f, indent=2, default=str)\n\n\ndef available_models(models_dir: str) -> tuple[dict, str]:\n    \"\"\"Return {display name: model} and the name of the backtest winner.\n\n    The saved XGBoost model is always included; Chronos-2 variants are\n    included when the `chronos` package is installed (weights load on first use).\n    \"\"\"\n    import importlib.util\n    import os\n    models = {}\n    prefix = os.path.join(models_dir, \"xgb_positive_cases\")\n    if os.path.exists(prefix + \".json\"):\n        models[MODEL_NAMES[\"xgb\"]] = LevelRelativeXGB.load(prefix)\n    if importlib.util.find_spec(\"chronos\") is not None:\n        models[MODEL_NAMES[\"chronos\"]] = ChronosForecaster(use_covariates=False)\n        models[MODEL_NAMES[\"chronos_cov\"]] = ChronosForecaster(use_covariates=True)\n    best = MODEL_NAMES[\"xgb\"]\n    path = os.path.join(models_dir, \"best_model.json\")\n    if os.path.exists(path):\n        with open(path) as f:\n            best = json.load(f).get(\"best\", best)\n    if best not in models and models:\n        best = next(iter(models))\n    return models, best\n",
 "app.py": "\"\"\"\nStreamlit app: view forecasts of daily positive COVID-19 cases.\n\nRun:  streamlit run app.py   ->  http://localhost:8501\n\nPick a history window, a forecast cut-off, a horizon and a model (the default\nis the model that won the notebook's backtest). If the cut-off is earlier than\nthe last complete date, the forecast only uses data up to the cut-off - XGBoost\nis re-trained on that data, Chronos-2 reads it as context - and is plotted\nagainst what actually happened.\n\"\"\"\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport plotly.graph_objects as go\nimport streamlit as st\n\nimport pipeline as pl\n\nBASE = os.path.dirname(os.path.abspath(__file__))\nMODELS_DIR = os.path.join(BASE, \"models\")\nDAILY_CSV = os.path.join(MODELS_DIR, \"daily_series.csv\")\nMETRICS = os.path.join(MODELS_DIR, \"metrics.json\")\nRAW_XLSX = os.path.join(BASE, \"data\", \"covid_data.xlsx\")\n\nst.set_page_config(page_title=\"COVID-19 Case Forecast\", page_icon=\":chart_with_upwards_trend:\", layout=\"wide\")\n\n\n# ---------------------------------------------------------------- loading\n@st.cache_resource(show_spinner=\"Loading data and models…\")\ndef load():\n    \"\"\"Load artefacts saved by the notebook; rebuild from raw data if missing.\"\"\"\n    if os.path.exists(DAILY_CSV) and os.path.exists(METRICS):\n        daily = pd.read_csv(DAILY_CSV, parse_dates=[\"date\"], index_col=\"date\").asfreq(\"D\")\n        with open(METRICS) as f:\n            meta = json.load(f)\n        complete = pd.Timestamp(meta[\"last_complete_date\"])\n        params = meta.get(\"params\", {})\n        models, best = pl.available_models(MODELS_DIR)\n    else:  # fallback: run the pipeline directly and train XGBoost\n        df, _ = pl.clean_data(pl.read_patient_excel(RAW_XLSX)[0], verbose=False)\n        daily, complete = pl.build_daily(df)\n        params, meta = {}, {}\n        os.makedirs(MODELS_DIR, exist_ok=True)\n        pl.train_model(daily.loc[:complete, \"positive\"].astype(float), exog=daily).save(\n            os.path.join(MODELS_DIR, \"xgb_positive_cases\"))\n        models, best = pl.available_models(MODELS_DIR)\n    exog = daily[pl.EXOG_COLS] if set(pl.EXOG_COLS) <= set(daily.columns) else None\n    return daily, complete, exog, params, models, best, meta\n\n\nDAILY, COMPLETE, EXOG, PARAMS, MODELS, BEST, META = load()\nY = DAILY.loc[:COMPLETE, \"positive\"].astype(float)\nMIN_CUTOFF = pd.Timestamp(pl.TRAIN_START) + pd.Timedelta(days=21)\nXGB_NAME = pl.MODEL_NAMES[\"xgb\"]\n\n\n@st.cache_data(show_spinner=\"Forecasting…\", max_entries=256)\ndef run_forecast(model_name: str, cutoff: str, horizon: int):\n    \"\"\"Forecast (and, for XGBoost, the 1-day-ahead fit) from a given cut-off.\"\"\"\n    c = pd.Timestamp(cutoff)\n    hist = Y.loc[:c]\n    model = MODELS[model_name]\n    retrained = False\n    if isinstance(model, pl.LevelRelativeXGB):\n        if c < COMPLETE:  # honest backtest: re-train on data up to the cut-off only\n            model = pl.train_model(hist, exog=EXOG, features=model.features, **PARAMS)\n            retrained = True\n        fitted = pl.one_step_predictions(model, hist, exog=EXOG)\n    else:\n        fitted = None\n    fc = model.forecast(hist, horizon, exog=EXOG)\n    return fc, fitted, retrained\n\n\n# ---------------------------------------------------------------- controls\nst.title(\"Daily positive COVID-19 cases: forecast\")\nst.caption(\"Choose a history window, a forecast cut-off and a model. Pick an earlier cut-off to \"\n           \"backtest: the forecast then only uses data up to that date and is compared with what \"\n           \"actually happened.\")\n\nif \"cutoff\" not in st.session_state:\n    st.session_state.cutoff = COMPLETE.date()\n\n\ndef set_cutoff(d):\n    st.session_state.cutoff = d\n\n\nwith st.sidebar:\n    st.header(\"Settings\")\n    names = list(MODELS)\n    labels = {n: (f\"{n} ★ best in backtest\" if n == BEST else n) for n in names}\n    model_name = st.selectbox(\"Model\", names, index=names.index(BEST), format_func=labels.get)\n    start = st.date_input(\"Show history from\", value=max(DAILY.index[0], COMPLETE - pd.Timedelta(days=70)).date(),\n                          min_value=DAILY.index[0].date(), max_value=COMPLETE.date())\n    cutoff = st.date_input(\"Forecast cut-off (last known day)\", key=\"cutoff\",\n                           min_value=MIN_CUTOFF.date(), max_value=COMPLETE.date())\n    horizon = st.slider(\"Forecast horizon (days)\", 1, 28, 14)\n    st.caption(\"Shortcuts\")\n    st.button(\"Latest data\", on_click=set_cutoff, args=(COMPLETE.date(),), width=\"stretch\")\n    for weeks in (2, 4):\n        d = COMPLETE - pd.Timedelta(weeks=weeks)\n        if d >= MIN_CUTOFF:\n            st.button(f\"Backtest from {d.day} {d.strftime('%b')}\", on_click=set_cutoff,\n                      args=(d.date(),), width=\"stretch\")\n\ncutoff = pd.Timestamp(cutoff)\nstart = min(pd.Timestamp(start), cutoff - pd.Timedelta(days=7))\n\ntry:\n    fc, fitted, retrained = run_forecast(model_name, cutoff.date().isoformat(), horizon)\nexcept Exception as e:  # e.g. Chronos weights can't be downloaded\n    st.error(f\"{model_name} failed: {e}\")\n    st.stop()\n\n# ---------------------------------------------------------------- chart\nidx = pd.date_range(start, fc.index[-1], freq=\"D\")\nactual = DAILY[\"positive\"].reindex(idx)\nhist_mask, after_mask, inc_mask = idx <= cutoff, (idx > cutoff) & (idx <= COMPLETE), idx > COMPLETE\n\nfig = go.Figure()\nfig.add_bar(x=idx[hist_mask], y=actual[hist_mask], name=\"Actual (history)\", marker_color=\"#9b9a95\")\nfig.add_bar(x=idx[after_mask], y=actual[after_mask], name=\"Actual after cut-off\", marker_color=\"#3d3c39\")\nfig.add_bar(x=idx[inc_mask], y=actual[inc_mask], name=\"Incomplete (results pending)\", marker_color=\"#d9d8d3\")\nif fc[\"lo\"].notna().any():\n    fig.add_scatter(x=list(fc.index) + list(fc.index[::-1]), y=list(fc[\"hi\"]) + list(fc[\"lo\"][::-1]),\n                    fill=\"toself\", fillcolor=\"rgba(235,104,52,.18)\", line=dict(width=0),\n                    name=\"Forecast 10-90% range\", hoverinfo=\"skip\")\nif fitted is not None:\n    fw = fitted.loc[start:cutoff]\n    fig.add_scatter(x=fw.index, y=fw, name=\"Model fit (1-day ahead)\", mode=\"lines\",\n                    line=dict(color=\"#2a78d6\", width=2))\nfc_line = pd.concat([pd.Series([Y.loc[cutoff]], index=[cutoff]), fc[\"forecast\"]])\nfig.add_scatter(x=fc_line.index, y=fc_line, name=\"Forecast\", mode=\"lines+markers\",\n                line=dict(color=\"#eb6834\", width=2.5), marker=dict(size=5))\n# line and label drawn separately: add_vline(annotation_text=...) breaks on date axes\n# with older Plotly + newer pandas (Timestamp + int TypeError)\nfig.add_vline(x=cutoff, line_dash=\"dash\", line_color=\"#8a8984\")\nfig.add_annotation(x=cutoff, y=1, yref=\"paper\", text=\"cut-off\", showarrow=False,\n                   xanchor=\"left\", yanchor=\"bottom\", font=dict(color=\"#8a8984\"))\nfig.update_layout(barmode=\"overlay\", height=460, margin=dict(l=10, r=10, t=30, b=10),\n                  yaxis_title=\"Positive cases per day\", hovermode=\"x unified\",\n                  legend=dict(orientation=\"h\", y=1.08, x=0))\nst.plotly_chart(fig, width=\"stretch\")\nst.caption(f\"Data after {COMPLETE.date()} is incomplete (many tests still awaiting results), \"\n           \"so it is excluded from training and shown as faded bars.\")\n\n# ---------------------------------------------------------------- metrics\noverlap = fc.index[fc.index <= COMPLETE]\ncols = st.columns(4)\ncols[0].metric(f\"Forecast total ({horizon} days)\", f\"{fc['forecast'].sum():,.0f}\",\n               f\"{fc['forecast'].mean():.1f} per day on average\", delta_color=\"off\")\nif len(overlap):\n    err = Y.loc[overlap] - fc.loc[overlap, \"forecast\"]\n    mape = (err.abs() / Y.loc[overlap].clip(lower=1)).mean() * 100\n    cols[1].metric(\"Forecast error (MAE)\", f\"{err.abs().mean():.2f}\",\n                   f\"{mape:.1f}% MAPE over {len(overlap)} days\", delta_color=\"off\")\nelse:\n    cols[1].metric(\"Forecast error (MAE)\", \"—\", \"no complete actuals after the cut-off\", delta_color=\"off\")\nif fitted is not None and len(fitted.loc[start:cutoff]):\n    fw = fitted.loc[start:cutoff]\n    cols[2].metric(\"Fit error in window (MAE)\", f\"{(Y.loc[fw.index] - fw).abs().mean():.2f}\",\n                   \"1-day-ahead, history window\", delta_color=\"off\")\nelse:\n    cols[2].metric(\"Fit error in window (MAE)\", \"—\", \"not applicable (zero-shot model)\", delta_color=\"off\")\nif isinstance(MODELS[model_name], pl.LevelRelativeXGB):\n    how = f\"Re-trained on data up to {cutoff.date()}\" if retrained else f\"Trained on data up to {COMPLETE.date()}\"\nelse:\n    how = f\"Zero-shot, context up to {cutoff.date()}\"\ncols[3].metric(\"Model\", model_name, how, delta_color=\"off\")\n\n# ---------------------------------------------------------------- tables\nwith st.expander(\"Forecast table\", expanded=True):\n    tbl = pd.DataFrame({\"Day\": fc.index.strftime(\"%a\"), \"Forecast\": fc[\"forecast\"].round(1)}, index=fc.index.date)\n    if fc[\"lo\"].notna().any():\n        tbl[\"Low (10%)\"] = fc[\"lo\"].round(1)\n        tbl[\"High (90%)\"] = fc[\"hi\"].round(1)\n    act = DAILY[\"positive\"].reindex(fc.index).where(fc.index <= COMPLETE)\n    tbl[\"Actual\"] = act.values\n    tbl[\"Error\"] = (fc[\"forecast\"] - act).round(1).values\n    st.dataframe(tbl, width=\"stretch\")\n\nif META.get(\"model_comparison\"):\n    with st.expander(\"Notebook backtest: model comparison\"):\n        st.dataframe(pd.DataFrame(META[\"model_comparison\"]), width=\"stretch\", hide_index=True)\n",
 "requirements.txt": "pandas>=2.0\nnumpy\nopenpyxl\nxgboost>=2.0\nscikit-learn\nmatplotlib\nchronos-forecasting>=2.0\nstreamlit\nplotly\njupyter\n"
}
CURRENT_MARKER = {"pipeline.py": "class ChronosForecaster", "app.py": "import streamlit"}

os.makedirs(PROJECT, exist_ok=True)
for rel, text in SOURCES.items():
    dst = os.path.join(PROJECT, rel)
    # also replace pipeline.py / app.py from an older project zip (e.g. the Flask
    # version, or one without Chronos-2 / the extra daily inputs)
    outdated = (rel in CURRENT_MARKER and os.path.exists(dst)
                and CURRENT_MARKER[rel] not in open(dst, encoding="utf-8").read())
    if outdated or not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with open(dst, "w", encoding="utf-8") as f:
            f.write(text)
        print("  wrote", rel)
for d in ("data", "models"):
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)

In [ ]:
#@title 0.4 Choose the Excel file, validate it and switch into the project
DATA_DST = os.path.join(PROJECT, "data", "covid_data.xlsx")
# priority: Excel uploaded directly > Excel inside a data-only zip > Excel bundled in the project zip
candidates = direct_excels + zipped_excels
if candidates:
    chosen = candidates[0]
    if len(candidates) > 1:
        print("Several Excel files found; using", os.path.basename(chosen))
    shutil.copy(chosen, DATA_DST)
    print("Data file:", os.path.basename(chosen))
elif os.path.exists(DATA_DST):
    print("Data file: data/covid_data.xlsx from the project zip")
else:
    raise FileNotFoundError("No Excel data found. Re-run step 0.2 and upload covid_data.xlsx "
                            "(or covid_forecast.zip, which contains it).")

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
for m in ("pipeline", "app"):
    sys.modules.pop(m, None)      # pick up freshly uploaded code on re-runs
# stale model artefacts from the zip are replaced by retraining below
import pipeline as pl
_raw, _ = pl.read_patient_excel(DATA_DST)
print(f"OK: {len(_raw):,} rows, {len(_raw.columns)} columns. Working dir: {os.getcwd()}")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import inspect, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

import pipeline as pl

plt.rcParams.update({"figure.figsize": (12, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_columns", 30)
DATA = "data/covid_data.xlsx"

## 1. Load & profile the raw data

In [ ]:
raw, dictionary = pl.read_patient_excel(DATA)
print(raw.shape)
raw.head()

In [ ]:
dictionary.style.set_properties(**{"text-align": "left"}) if dictionary is not None else print("No data_dictionary sheet")

In [ ]:
# Missing values and the distinct values of each categorical column
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_missing": raw.isna().sum(),
    "pct_missing": (raw.isna().mean() * 100).round(1),
    "n_unique": raw.nunique(),
})
profile

### Data-quality checks
Each check below surfaces a specific error class in the raw file.

In [ ]:
print("sex values:", raw["sex"].value_counts().to_dict())

In [ ]:
# IDs should be 6-character hex strings. Excel has mangled some of them.
valid_id = raw["id"].astype(str).str.fullmatch(r"[0-9a-f]{6}")
print("invalid IDs:", (~valid_id).sum())
print("duplicate IDs:", raw["id"].duplicated().sum())
raw.loc[~valid_id, "id"].value_counts().head(10)

In [ ]:
print(raw["age"].describe())
print("\nimpossible ages:", sorted(raw.loc[(raw.age < 0) | (raw.age > 110), "age"].tolist()))

In [ ]:
d = {c: pd.to_datetime(raw[c], format="%d-%m-%Y", errors="coerce") for c in pl.DATE_COLS}
print("unparseable dates:", {c: int(d[c].isna().sum() - raw[c].isna().sum()) for c in d})
print("symptoms after entry:", int((d["date_symptoms"] > d["entry_date"]).sum()))
print("died before entry   :", int((d["date_died"] < d["entry_date"]).sum()))
raw.loc[d["date_died"] < d["entry_date"], ["id", "entry_date", "date_symptoms", "date_died"]]

In [ ]:
# Structural missingness: pregnancy for males, intubed/icu for outpatients
print(pd.crosstab(raw["sex"], raw["pregnancy"].fillna("NaN")))
print()
print(pd.crosstab(raw["patient_type"], raw["icu"].fillna("NaN")))

**Errors found** (counts are for the original `covid_data.xlsx`; the cleaning log in §2 shows the counts for whatever file you load)

| Issue | Rows | Fix |
|---|---|---|
| `sex` typos (`Femalee`, `Males`) | 163 | map to `Female` / `Male` |
| IDs lost leading zeros (e.g. `50969` → `050969`) | 477 | zero-pad to 6 chars |
| IDs converted to scientific notation (`1.34E+08`, `0.00E+00`) — causes the "duplicate" IDs | 242 | original value unrecoverable; rows are distinct patients, so keep them and assign a synthetic `genNNNNN` id |
| Impossible ages (−67, 856, 483 …) | 8 | set to missing → impute median, flag in `age_was_invalid` |
| `date_died` earlier than `entry_date` | 2 | logically inconsistent → null, flag in `date_inconsistent` |
| Dates stored as `dd-mm-yyyy` text | all | parse to datetime |
| `pregnancy` blank for males, `intubed`/`icu` blank for outpatients | ~5k / ~7.9k | *not applicable*, not unknown → 0 |
| Small gaps in comorbidities (~0.3% each) | 375 cells | fill with mode (`No`) |
| `contact_other_covid` 31% missing | 3,107 | too many to impute → explicit "unknown" (−1) |
| `covid_res` = `Awaiting Results` | 1,122 | not a positive; drives the reporting-lag handling in §3 |

## 2. Cleaning

In [ ]:
print(inspect.getsource(pl.clean_data))

In [ ]:
df, log = pl.clean_data(raw)

In [ ]:
# Verify the cleaned data
assert df["sex"].isin(["Male", "Female"]).all()
assert df["id"].is_unique
assert df["age"].between(0, 110).all()
assert not (df["date_died"] < df["entry_date"]).any()
assert df[pl.COMORBIDITIES].isna().sum().sum() == 0
print("All cleaning checks passed. Shape:", df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
df["age"].plot.hist(bins=40, ax=ax[0], color="#4C72B0", title="Age (cleaned)")
df["covid_res"].value_counts().plot.bar(ax=ax[1], color="#4C72B0", title="Test result", rot=0)
(df.groupby("covid_res", observed=True)[pl.COMORBIDITIES].mean().T * 100).plot.barh(
    ax=ax[2], title="Comorbidity prevalence (%) by result")
plt.tight_layout()

## 3. Daily time series

The target is **daily positive cases by `entry_date`**. One subtlety: the last days of the file show a collapse in positives (e.g. 0 on 28 June). That isn't a real drop: most of those tests are still *Awaiting Results*. Training on those days would teach the model a fake decline, so the series is cut at the last date where ≤25% of tests are still pending.

In [ ]:
daily, complete = pl.build_daily(df)
print("Last complete date:", complete.date())
daily.tail(10)

In [ ]:
fig, ax = plt.subplots()
ax.bar(daily.index, daily["positive"], color="#4C72B0", label="Positive cases")
ax.plot(daily.index, daily["positive"].rolling(7).mean(), color="#DD8452", lw=2, label="7-day mean")
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(daily.index, daily["awaiting_share"] * 100, color="grey", ls=":", label="% awaiting results")
ax2.set_ylabel("% awaiting")
ax.axvspan(complete + pd.Timedelta(hours=12), daily.index[-1] + pd.Timedelta(days=1), color="red", alpha=.08)
ax.text(complete + pd.Timedelta(days=1), ax.get_ylim()[1] * .9, "incomplete\n(excluded)", color="red", fontsize=9)
ax.set_title("Daily positive COVID-19 cases"); ax.legend(loc="upper left")
plt.show()

In [ ]:
y = daily.loc[:complete, "positive"].astype(float)
recent = y.iloc[-70:]
dow = recent.groupby(recent.index.day_name()).mean()
dow = dow.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
dow.plot.bar(color="#4C72B0", rot=0, title="Mean daily positives by weekday (last 10 weeks)", figsize=(8,3))
plt.show()

There is a strong **weekly cycle** — weekends record roughly half the weekday count — so day-of-week and 7-day lags are essential features.

### Other daily counts used as model inputs
Besides past positives, the model now also sees three other daily counts by entry date: tests **still pending**, patients **reporting contact** with another COVID case, and **admissions** (patients who are not outpatients).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2), sharex=True)
for ax, col, title in zip(axes, pl.EXOG_COLS, ["Pending results", "Reporting contact", "Admissions"]):
    ax.bar(daily.index, daily[col], color="#4C72B0", width=1)
    ax.plot(daily.index, daily[col].rolling(7).mean(), color="#DD8452", lw=2)
    ax.axvline(complete, color="red", lw=.8)
    ax.set_title(title); ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()
print("Correlation with same-day positives (complete days):")
daily.loc[:complete, ["positive"] + pl.EXOG_COLS].corr()["positive"].drop("positive").round(2)

## 4. Feature engineering

In [ ]:
print(inspect.getsource(pl.make_features))
EXOG = daily[pl.EXOG_COLS]          # pending results, contacts, admissions per day
X_all = pl.make_features(y, EXOG)
print("Features:", len(pl.FEATURES), "=", len(pl.BASE_FEATURES), "base +", len(pl.EXOG_FEATURES), "new")
X_all.dropna().tail()

**New features** (each uses yesterday's value and the 7-day mean up to yesterday, so nothing from the day being predicted leaks in):

| Feature | Meaning |
|---|---|
| `awaiting_lag_1`, `awaiting_roll_7` | tests still pending |
| `contacts_lag_1`, `contacts_roll_7` | patients reporting contact with another COVID case |
| `admissions_lag_1`, `admissions_roll_7` | patients admitted (not outpatients) |

Two things to keep in mind:
* **Future values are unknown.** For a 14-day forecast, days 2-14 need these counts for days that haven't happened yet. `recursive_forecast` fills them with the value from the same weekday a week earlier.
* **Pending results are a snapshot.** The file only records how many tests were still pending *when it was exported*, not how many were pending on each past day. On older days almost everything is resolved, while near the cut-off up to 25% is pending, so this feature looks different at forecast time than in training, and the backtest can't fully test it. The ablation below shows whether it helps.

## 5. Modelling

### Why a level-relative target?
Decision trees can't predict values outside the range they saw in training. On a rising epidemic curve the test period often exceeds anything in the training data, so a plain XGBoost regressor saturates. `LevelRelativeXGB` instead predicts `log1p(y) − log1p(7-day mean)`, a scale-free target, then rescales. Multi-day forecasts are **recursive**: each prediction is fed back in as a lag for the next day.

In [ ]:
print(inspect.getsource(pl.LevelRelativeXGB))

### Rolling-origin backtest
A single train/test split on ~100 days is noisy, so the models are compared over 5 forecast origins (weekly), each forecasting 14 days ahead using only data up to that origin.

Models compared:
* **Seasonal naive** — repeat the last observed week
* **XGBoost (raw target)** — standard `XGBRegressor` on counts
* **XGBoost (level-relative)** — original 11 features
* **XGBoost (level-relative + new)** — plus pending results, contacts and admissions
* **Chronos-2** — pretrained foundation model, zero-shot (no training on this data)
* **Chronos-2 + covariates** — Chronos-2 also given pending results, contacts and admissions as past-only inputs

In [ ]:
#@title Load Chronos-2 (downloads the weights from Hugging Face on first run)
USE_CHRONOS = True  #@param {type:"boolean"}
CHRONOS = None
if USE_CHRONOS:
    try:
        CHRONOS = pl.load_chronos()
        print("Chronos-2 loaded:", pl.CHRONOS_MODEL_ID)
    except Exception as e:
        print("Chronos-2 not available, comparing XGBoost models only.\n ", type(e).__name__, e)

In [ ]:
class RawXGB:
    def __init__(self, features=None, **p):
        self.features = features or pl.BASE_FEATURES
        self.m = xgb.XGBRegressor(**{**pl.DEFAULT_PARAMS, **p})
    def fit(self, X, y): self.m.fit(X[self.features], y); return self
    def predict(self, X): return np.clip(self.m.predict(X[self.features]), 0, None)

def fit_on(y_train, cls, features, **p):
    # every model trains on the same rows; exog is cut at the origin inside make_features
    X = pl.make_features(y_train, EXOG).dropna().loc[pl.TRAIN_START:]
    return cls(features=features, **p).fit(X, y_train.loc[X.index])

def fc_at(o, cls, features, **p):
    tr = y[:o]
    return pl.recursive_forecast(fit_on(tr, cls, features, **p), tr, H, exog=EXOG)

H = 14
origins = pd.date_range(end=complete - pd.Timedelta(days=H), periods=5, freq="7D")
rows, paths = [], {}
for o in origins:
    tr = y[:o]; te = y[o + pd.Timedelta(days=1): o + pd.Timedelta(days=H)]
    preds = {
        "Seasonal naive": pd.Series(np.tile(tr[-7:].values, 3)[:H], index=te.index),
        "XGBoost (raw)": fc_at(o, RawXGB, pl.BASE_FEATURES),
        "XGBoost (level-relative)": fc_at(o, pl.LevelRelativeXGB, pl.BASE_FEATURES),
        "XGBoost (level-relative + new)": fc_at(o, pl.LevelRelativeXGB, pl.FEATURES),
    }
    if CHRONOS is not None:
        for key, cov in (("chronos", False), ("chronos_cov", True)):
            f = pl.ChronosForecaster(use_covariates=cov, pipeline=CHRONOS).forecast(tr, H, exog=EXOG)
            preds[pl.MODEL_NAMES[key]] = f["forecast"]
    paths[o] = preds
    for name, p in preds.items():
        rows.append({"origin": o.date(), "model": name,
                     "MAE": mean_absolute_error(te, p),
                     "RMSE": np.sqrt(mean_squared_error(te, p))})
bt = pd.DataFrame(rows)
bt.pivot(index="origin", columns="model", values="MAE").round(1)

In [ ]:
summary = bt.groupby("model")[["MAE", "RMSE"]].mean().sort_values("MAE").round(2)
summary

In [ ]:
fig, axes = plt.subplots(1, len(origins), figsize=(18, 3.5), sharey=True)
for ax, o in zip(axes, origins):
    win = y[o - pd.Timedelta(days=20): o + pd.Timedelta(days=H)]
    ax.plot(win.index, win, color="black", lw=1.5, label="Actual")
    for (name, p), c in zip(paths[o].items(), ["grey", "#DD8452", "#4C72B0", "#55A868", "#8172B3", "#C44E52"]):
        ax.plot(p.index, p, color=c, lw=1.5, ls="--" if name == "Seasonal naive" else "-", label=name)
    ax.axvline(o, color="red", lw=.8); ax.set_title(f"Origin {o.date()}", fontsize=9)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
axes[0].legend(fontsize=7); plt.tight_layout()

### Which of the new inputs help? (ablation)
Same backtest, level-relative model, adding each group of new features on its own and then all three together. Lower MAE is better.

In [ ]:
groups = {"pending results": "awaiting", "contacts": "contacts", "admissions": "admissions"}
feature_sets = {"original 11": pl.BASE_FEATURES}
for label, col in groups.items():
    feature_sets[f"+ {label}"] = pl.BASE_FEATURES + [f for f in pl.EXOG_FEATURES if f.startswith(col)]
feature_sets["+ all three"] = pl.FEATURES

abl = []
for name, feats in feature_sets.items():
    maes = [mean_absolute_error(y[o + pd.Timedelta(days=1): o + pd.Timedelta(days=H)],
                                fc_at(o, pl.LevelRelativeXGB, feats)) for o in origins]
    abl.append({"feature set": name, "n_features": len(feats), "MAE": np.mean(maes),
                "MAE vs original": np.mean(maes) - abl[0]["MAE"] if abl else 0.0})
abl = pd.DataFrame(abl).set_index("feature set").round(2)
abl

In [ ]:
#@title Feature set for the final model
FEATURE_SET = "all three"  #@param ["all three", "best in ablation", "original 11"]
if FEATURE_SET == "best in ablation":
    FINAL_FEATURES = feature_sets[abl["MAE"].idxmin()]
elif FEATURE_SET == "original 11":
    FINAL_FEATURES = pl.BASE_FEATURES
else:
    FINAL_FEATURES = pl.FEATURES
print(f"Final model uses {len(FINAL_FEATURES)} features:", FINAL_FEATURES)

### Light hyper-parameter check (same backtest)

In [ ]:
grid = [dict(max_depth=d, n_estimators=n, learning_rate=lr)
        for d in (2, 3) for n in (200, 300, 500) for lr in (0.03, 0.05)]
res = []
for g in grid:
    maes = []
    for o in origins:
        tr = y[:o]; te = y[o + pd.Timedelta(days=1): o + pd.Timedelta(days=H)]
        maes.append(mean_absolute_error(te, fc_at(o, pl.LevelRelativeXGB, FINAL_FEATURES, **g)))
    res.append({**g, "MAE": np.mean(maes)})
grid_res = pd.DataFrame(res).sort_values("MAE")
grid_res.head(6).round(2)

In [ ]:
best = grid_res.iloc[0][["max_depth", "n_estimators", "learning_rate"]].to_dict()
best = {"max_depth": int(best["max_depth"]), "n_estimators": int(best["n_estimators"]),
        "learning_rate": float(best["learning_rate"])}
print("Selected params:", best)

## 6. Pick the best model
The tuned XGBoost model and the two Chronos-2 variants are compared on the same five backtest origins, and the one with the lowest MAE is served by the web app. Seasonal naive is shown for reference only.

Note that XGBoost's settings were tuned on these same origins, which gives it a small advantage; Chronos-2 wasn't tuned at all.

In [ ]:
def score(pred_at):
    m, r = [], []
    for o in origins:
        te = y[o + pd.Timedelta(days=1): o + pd.Timedelta(days=H)]
        p = pred_at(o)
        m.append(mean_absolute_error(te, p)); r.append(np.sqrt(mean_squared_error(te, p)))
    return np.mean(m), np.mean(r)

candidates = {pl.MODEL_NAMES["xgb"]: lambda o: fc_at(o, pl.LevelRelativeXGB, FINAL_FEATURES, **best)}
for key in ("chronos", "chronos_cov"):
    name = pl.MODEL_NAMES[key]
    if name in paths[origins[0]]:
        candidates[name] = lambda o, name=name: paths[o][name]

rows = []
for name, pred_at in candidates.items():
    mae, rmse = score(pred_at)
    rows.append({"model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "selectable": True})
mae, rmse = score(lambda o: paths[o]["Seasonal naive"])
rows.append({"model": "Seasonal naive (reference)", "MAE": round(mae, 2), "RMSE": round(rmse, 2), "selectable": False})
comparison = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
BEST_MODEL = comparison[comparison.selectable].iloc[0]["model"]
print("Best model:", BEST_MODEL)
comparison.drop(columns="selectable")

## 7. Final model, 14-day forecast & artefacts

In [ ]:
model = pl.train_model(y, exog=EXOG, features=FINAL_FEATURES, **best)
fitted = pl.one_step_predictions(model, y, exog=EXOG).loc[pl.TRAIN_START:]
forecast = pl.recursive_forecast(model, y, 14, exog=EXOG)

# forecast from the backtest winner (the XGBoost model above is always saved too)
if BEST_MODEL == pl.MODEL_NAMES["xgb"]:
    best_model = model
else:
    best_model = pl.ChronosForecaster(use_covariates=BEST_MODEL == pl.MODEL_NAMES["chronos_cov"], pipeline=CHRONOS)
best_fc = best_model.forecast(y, 14, exog=EXOG)

print(f"In-sample one-step MAE: {mean_absolute_error(y.loc[fitted.index], fitted):.2f}")
print("Best model:", BEST_MODEL)
best_fc.round(1).T

In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y, color="black", lw=1.2, label="Actual")
ax.plot(fitted.index, fitted, color="#4C72B0", alpha=.7, label="Fitted (1-day ahead)")
ax.plot(best_fc.index, best_fc["forecast"], color="#DD8452", lw=2, marker="o", ms=3, label=f"Forecast: {BEST_MODEL}")
if best_fc["lo"].notna().any():
    ax.fill_between(best_fc.index, best_fc["lo"], best_fc["hi"], color="#DD8452", alpha=.2, label="10-90% range")
inc = daily.loc[complete + pd.Timedelta(days=1):, "positive"]
ax.scatter(inc.index, inc, color="red", s=12, label="Incomplete (excluded)")
ax.set_title("Daily positive cases — XGBoost forecast"); ax.legend()
plt.show()

In [ ]:
# XGBoost feature importance (Chronos-2 has no equivalent)
model.feature_importances_.sort_values().plot.barh(color="#4C72B0", figsize=(7, 4),
                                                    title="Feature importance (gain share)")
plt.show()

In [ ]:
import os
os.makedirs("models", exist_ok=True)
model.save("models/xgb_positive_cases")
daily.to_csv("models/daily_series.csv")
df.to_csv("data/covid_data_clean.csv", index=False)
with open("models/metrics.json", "w") as f:
    json.dump({"last_complete_date": str(complete.date()), "params": best,
               "features": FINAL_FEATURES,
               "backtest": summary.reset_index().to_dict(orient="records"),
               "feature_ablation": abl.reset_index().to_dict(orient="records"),
               "model_comparison": comparison.drop(columns="selectable").to_dict(orient="records"),
               "best_model": BEST_MODEL}, f, indent=2, default=str)
pl.save_best("models", BEST_MODEL, comparison.drop(columns="selectable").to_dict(orient="records"))
print(os.listdir("models"))

## Conclusions
* The raw file had several error classes (typos, Excel-corrupted IDs, impossible ages, inconsistent dates, structural blanks), all fixed and flagged rather than silently dropped. No patient rows were removed.
* The last ~6 days are artificially low because results were still pending; they're excluded from training and shown separately.
* A plain XGBoost regressor barely beats repeating last week because trees can't extrapolate a rising trend. Modelling the **log-ratio to the 7-day level** fixes this and gives the lowest backtest error.
* The weekend / day-of-week features carry most of the importance, followed by the weekly lags (7, 14) and recent volatility, matching the strong weekly reporting cycle.
* The model can also use three other daily counts: pending results, patients reporting contact, and admissions (yesterday's value and 7-day mean of each). The ablation table in §5 shows how much each changes the backtest error on your data; with ~100 days, differences of a case or two per day are within noise. Pending results are recorded as of the export date, so treat that feature with extra caution.
* **Chronos-2**, a pretrained forecasting model, is compared with XGBoost on the same backtest, with and without the extra daily inputs. The model with the lowest backtest MAE (table in §6) is saved as the default for the web app; the others stay selectable there.
* **Caveat:** ~100 days of history is small; treat forecasts beyond a week or two as indicative. The Streamlit app (`app.py`) lets you pick a cut-off date and horizon to see how the model performs against actuals.

## 8. Download the results

In [ ]:
#@title 8. Zip the project (model, cleaned data, notebook outputs) and download it
out_zip = shutil.make_archive(os.path.join(WORK, "covid_forecast_results"), "zip",
                              root_dir=WORK, base_dir="covid_forecast")
print("Created", out_zip)
if IN_COLAB:
    from google.colab import files
    files.download(out_zip)

## 9. Web page (Streamlit + localtunnel)
The second cell below starts Streamlit on port 8501, opens a public **localtunnel** link to that port, and prints this machine's public IP address.

1. Wait for the line `your url is: https://….loca.lt` and click it.
2. localtunnel shows a reminder page asking for a **tunnel password**. Enter the IP address printed by `curl ipv4.icanhazip.com` (a line like `34.12.xx.xx`) and click *Submit*.
3. The forecast page loads. The model that won the backtest is selected by default; the others are in the **Model** dropdown in the sidebar.

The cell keeps running while the web app is up; that's expected, which is why this is the last step. Stop it with the ■ button. If you re-run it, the first cell stops the previous server so port 8501 is free. Streamlit's own log is in `streamlit.log`.

In [ ]:
import os
os.chdir(PROJECT)                     # app.py, pipeline.py, models/ live here
!pkill -f "[s]treamlit run" ; pkill -f "[l]ocaltunnel" ; true   # stop a previous run, frees port 8501

In [ ]:
# Start Streamlit, open the tunnel, print the tunnel password (this machine's IP)
!streamlit run app.py --server.port 8501 --server.headless true --browser.gatherUsageStats false --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 & npx localtunnel --port 8501 & curl ipv4.icanhazip.com